# FMCW 레이더(VoD) — 데이터·전처리·DBSCAN·다중센서 검증·연속프레임 추적·위험도 (상세 구현)

본 노트북은 **프로젝트 세부 수행 사항** 중 **FMCW 레이더 / VoD Multi-Sensor / 지도 시각화**에 해당하는 부분을 **코드와 수식·해설**로 정리한 것입니다.

**전제:**
- View-of-Delft **KITTI 트리** 레이아웃 (`radar/training/velodyne/*.bin`, `lidar/training/velodyne/*.bin`, `lidar/training/image_2/*.jpg`).
- 레이더 `.bin` 한 점당 **7×float32**: `x, y, z, RCS, v_r, v_r_comp, time` (ego/차량 좌표계, **+x 전방** 근사).
- **학습용 NN 가중치 없음**: 기하 **DBSCAN**, **그리디 연관**, **규칙 기반 위험도** — 백엔드 `ai-inference/main.py`의 `POST /infer/vod/radar-fusion`과 **동일 계열**입니다.

**의존성:** `numpy`, `matplotlib`, `scikit-learn` (필수) · `pandas` (선택, 표 출력)

## 목차

1. [데이터 스키마 및 로드](#schema)  
2. [전처리: 이상치·ROI·속도 채널](#pre)  
3. [DBSCAN 표적 후보 및 신뢰도](#dbscan)  
4. [후보 필터링(오탐 억제)](#filter)  
5. [LiDAR 기하 검증(동기 프레임)](#lidar)  
6. [연속 프레임 연관·속도·heading·8방위](#track)  
7. [추적 안정화(게이트·간단 스무딩)](#smooth)  
8. [규칙 기반 위험도 및 외삽 궤적](#risk)  
9. [시각화(BEV) 및 요약](#viz)

In [ ]:
# %pip install -q numpy matplotlib scikit-learn
# %pip install -q pandas  # 선택

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

try:
    import pandas as pd
except ImportError:
    pd = None

# 한글 폰트 (Windows)
from matplotlib import font_manager, rc

_fp = Path("C:/Windows/Fonts/malgun.ttf")
if _fp.is_file():
    _fn = font_manager.FontProperties(fname=str(_fp)).get_name()
    rc("font", family=_fn)
    rc("axes", unicode_minus=False)

<a id="schema"></a>
## 1. 데이터 스키마 및 로드

| 필드 | 의미 | 활용 |
|------|------|------|
| x,y,z | ego 기준 위치 (m) | 군집화·거리·방위 |
| RCS | 산란 강도 스칼라 | 후보 신뢰도 보조 |
| v_r | 상대 방사속도 성분 | 물리적 해석(접근/이탈) |
| v_r_comp | 자차 보정 속도 | 정적 배경 vs 이동체 구분에 활용 |
| time | 타임스탬프 | (옵션) 스캔 정렬 |

**누적 스캔:** VoD는 single / 3-scan / 5-scan 등이 있을 수 있으나, 본 노트북은 **프레임당 단일 `.bin`**을 가정합니다. 누적 데이터는 동일 파이프라인에 **더 많은 점**으로 입력하면 됩니다.

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
DEFAULT_ROOT = NOTEBOOK_DIR / "vod-received" / "view_of_delft_PUBLIC"
ROOT_DIR = Path(os.environ.get("VOD_ROOT", str(DEFAULT_ROOT))).resolve()

RADAR_DIR = ROOT_DIR / "radar" / "training" / "velodyne"
LIDAR_DIR = ROOT_DIR / "lidar" / "training" / "velodyne"


def list_synced_stems(root: Path) -> list[str]:
    cam = root / "lidar" / "training" / "image_2"
    rad = root / "radar" / "training" / "velodyne"
    if not cam.is_dir() or not rad.is_dir():
        return []
    jpgs = {p.stem for p in cam.glob("*.jpg")}
    bins = {p.stem for p in rad.glob("*.bin")}
    common = sorted(jpgs & bins, key=lambda s: (0, int(s)) if s.isdigit() else (1, s))
    return common


def parse_vod_radar_bin(path: Path) -> np.ndarray:
    """(N, 7) float32 — [x,y,z, RCS, v_r, v_r_comp, time]"""
    raw = np.fromfile(path, dtype=np.float32)
    if raw.size % 7 != 0:
        raise ValueError(f"7의 배수 아님: {path}")
    return raw.reshape(-1, 7)


def parse_vod_lidar_bin(path: Path) -> np.ndarray:
    """(N, 4) float32 — [x,y,z, intensity]"""
    raw = np.fromfile(path, dtype=np.float32)
    if raw.size % 4 != 0:
        raise ValueError(f"4의 배수 아님: {path}")
    return raw.reshape(-1, 4)


stems = list_synced_stems(ROOT_DIR)
print("ROOT_DIR:", ROOT_DIR, "exists:", ROOT_DIR.is_dir())
print("동기 프레임 수:", len(stems))
if len(stems) < 2:
    print("[경고] 연속 프레임 실험을 위해 최소 2개 이상의 stem이 필요합니다.")
else:
    FRAME_IDX = int(os.environ.get("VOD_FRAME_IDX", "1"))  # 0이면 이전 프레임 없음
    FRAME_IDX = max(1, min(FRAME_IDX, len(stems) - 1))
    stem_curr = stems[FRAME_IDX]
    stem_prev = stems[FRAME_IDX - 1]
    print("stem_prev:", stem_prev, "| stem_curr:", stem_curr)

In [ ]:
if len(stems) >= 2:
    path_r_curr = RADAR_DIR / f"{stem_curr}.bin"
    path_r_prev = RADAR_DIR / f"{stem_prev}.bin"
    path_l_curr = LIDAR_DIR / f"{stem_curr}.bin"
    pts_curr = parse_vod_radar_bin(path_r_curr)
    pts_prev = parse_vod_radar_bin(path_r_prev)
    lid_curr = parse_vod_lidar_bin(path_l_curr) if path_l_curr.is_file() else None
else:
    pts_curr = np.zeros((0, 7), dtype=np.float32)
    pts_prev = pts_curr
    lid_curr = None
    stem_curr = stem_prev = ""

print("현재 프레임 레이더 점 수:", pts_curr.shape[0])
print("직전 프레임 레이더 점 수:", pts_prev.shape[0])
if lid_curr is not None:
    print("동기 LiDAR 점 수:", lid_curr.shape[0])

<a id="pre"></a>
## 2. 데이터 전처리 (FMCW 포인트클라우드)

문서 요지 반영:
- **과도한 거리·비정상 높이·희소 반사** 제거 → clutter/ghost 완화  
- **전방 ROI** (x>0 구간 등)로 관심 영역 축소  
- **v_r / v_r_comp** 분리: 정적 배경은 보정 후 속도가 0에 가깝고, 이동체는 상대 성분이 남는 경향(휴리스틱)

아래는 **데모용 임계값**이며, `os.environ`로 조정 가능하게 했습니다.

In [ ]:
MAX_RANGE_M = float(os.environ.get("VOD_PRE_MAX_RANGE", "200.0"))
MIN_X_M = float(os.environ.get("VOD_PRE_MIN_X", "0.0"))  # 후방 제거: 전방만
Z_ABS_MAX = float(os.environ.get("VOD_PRE_Z_ABS_MAX", "8.0"))
MIN_RCS = float(os.environ.get("VOD_PRE_MIN_RCS", "-30.0"))  # 너무 약한 반사 제거(데이터에 맞게 조정)


def preprocess_radar(pts: np.ndarray) -> tuple[np.ndarray, dict]:
    """pts: (N,7) → 필터된 (N',7), 통계 meta"""
    if pts.size == 0:
        return pts, {"n_in": 0, "n_out": 0}
    n_in = pts.shape[0]
    xyz = pts[:, :3]
    rcs = pts[:, 3]
    rng = np.linalg.norm(xyz, axis=1)
    z = xyz[:, 2]
    x = xyz[:, 0]
    m = (
        (rng <= MAX_RANGE_M)
        & (np.abs(z) <= Z_ABS_MAX)
        & (x >= MIN_X_M)
        & (rcs >= MIN_RCS)
    )
    out = pts[m]
    meta = {
        "n_in": int(n_in),
        "n_out": int(out.shape[0]),
        "removed": int(n_in - out.shape[0]),
    }
    return out, meta


if pts_curr.shape[0] > 0:
    pts_curr_f, meta_curr = preprocess_radar(pts_curr)
    pts_prev_f, meta_prev = preprocess_radar(pts_prev)
    print("현재 프레임 전처리:", meta_curr)
    print("직전 프레임 전처리:", meta_prev)
else:
    pts_curr_f = pts_curr
    pts_prev_f = pts_prev

<a id="dbscan"></a>
## 3. DBSCAN 기반 표적 후보 추출

**입력 특징:** 3D 위치 `xyz`만 사용 (기하 군집).  
**출력:** 클러스터별 중심(centroid), $\text{range}=\|c\|$, $\text{azimuth}=\mathrm{atan2}(c_y,c_x)$, $\text{elevation}=\mathrm{atan2}(c_z,\sqrt{c_x^2+c_y^2})$, 평균 `v_r_comp`(도플러 근사), 평균 RCS, **신뢰도** heuristic.

$$
\mathrm{conf} = \min\Big(0.99,\; 0.25 + 0.02\min(n,20) + 0.15\min(|\bar v|/8,1) + 0.1\min(\overline{\mathrm{RCS}}/30,1)\Big)
$$

In [ ]:
DBSCAN_EPS = float(os.environ.get("VOD_RADAR_DBSCAN_EPS", "4.0"))
DBSCAN_MIN_SAMPLES = int(os.environ.get("VOD_RADAR_DBSCAN_MIN_SAMPLES", "3"))
MAX_CLUSTERS = int(os.environ.get("VOD_MAX_CLUSTERS", "12"))


def radar_clusters_dbscan(pts: np.ndarray, eps: float, min_samples: int, max_clusters: int = 12) -> list[dict]:
    xyz = pts[:, :3]
    rcs = pts[:, 3]
    v_comp = pts[:, 5]
    if xyz.shape[0] == 0:
        return []
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(xyz)
    labels = clustering.labels_
    out: list[dict] = []
    for lab in sorted(set(labels.tolist())):
        if lab < 0:
            continue
        m = labels == lab
        c = xyz[m].mean(axis=0)
        rng = float(np.linalg.norm(c))
        azimuth_deg = float(np.degrees(np.arctan2(c[1], c[0])))
        elevation_deg = float(np.degrees(np.arctan2(c[2], np.hypot(c[0], c[1]) + 1e-6)))
        vd = v_comp[m]
        doppler_mps = float(np.mean(vd)) if vd.size else 0.0
        rc = rcs[m]
        rcs_mean = float(np.mean(rc)) if rc.size else 0.0
        npts = int(m.sum())
        conf = min(
            0.99,
            0.25 + 0.02 * min(npts, 20) + 0.15 * min(abs(doppler_mps) / 8.0, 1.0) + 0.1 * min(rcs_mean / 30.0, 1.0),
        )
        out.append(
            {
                "id": f"cluster-{lab}",
                "label": int(lab),
                "rangeM": round(rng, 3),
                "azimuthDeg": round(azimuth_deg, 3),
                "elevationDeg": round(elevation_deg, 3),
                "dopplerMps": round(doppler_mps, 4),
                "confidence": round(conf, 4),
                "clusterSize": npts,
                "centroidM": [float(c[0]), float(c[1]), float(c[2])],
            }
        )
    out.sort(key=lambda d: d["confidence"], reverse=True)
    return out[:max_clusters]


dets_curr = radar_clusters_dbscan(pts_curr_f, DBSCAN_EPS, DBSCAN_MIN_SAMPLES, MAX_CLUSTERS)
dets_prev = radar_clusters_dbscan(pts_prev_f, DBSCAN_EPS, DBSCAN_MIN_SAMPLES, MAX_CLUSTERS)
print("현재 프레임 후보 수:", len(dets_curr))
for i, d in enumerate(dets_curr[:5]):
    print(i, d["id"], "r=", d["rangeM"], "az=", d["azimuthDeg"], "conf=", d["confidence"], "n=", d["clusterSize"])

<a id="filter"></a>
## 4. 후보 필터링 (오탐 억제)

- 최소 점 수 `MIN_PTS`  
- 거리 범위 `[R_MIN, R_MAX]`  
- (옵션) |azimuth| 너무 큰 측면 클러스터 제외 등 — 시나리오별 확장

In [ ]:
MIN_PTS = int(os.environ.get("VOD_FILTER_MIN_PTS", "4"))
R_MIN = float(os.environ.get("VOD_FILTER_R_MIN", "3.0"))
R_MAX = float(os.environ.get("VOD_FILTER_R_MAX", "150.0"))


def filter_detections(dets: list[dict]) -> list[dict]:
    out = []
    for d in dets:
        if d["clusterSize"] < MIN_PTS:
            continue
        if not (R_MIN <= d["rangeM"] <= R_MAX):
            continue
        out.append(d)
    return out


dets_curr_f = filter_detections(dets_curr)
dets_prev_f = filter_detections(dets_prev)
print("필터 후 현재 후보:", len(dets_curr_f), "/", len(dets_curr))

<a id="lidar"></a>
## 5. LiDAR 기하 검증 (동기 프레임)

레이더 클러스터 중심 $c$ 반경 $R$ 내 LiDAR 점 수·중심과의 $\Delta$range, $\Delta$방위로 **일치/부분/불일치** 판정.  
백엔드 `_lidar_validate_cluster`와 동일 로직입니다.

In [ ]:
LIDAR_ROI_R = float(os.environ.get("VOD_LIDAR_ROI_M", "2.5"))


def bearing_deg_xy(x: float, y: float) -> float:
    return float(np.degrees(np.arctan2(y, x)))


def angle_diff_abs_deg(a: float, b: float) -> float:
    d = (a - b + 180.0) % 360.0 - 180.0
    return abs(d)


def lidar_validate_cluster(
    lidar_xyz: np.ndarray,
    centroid: list[float],
    radius_m: float,
    radar_range_m: float | None,
    radar_azimuth_deg: float | None,
) -> dict:
    c = np.array(centroid, dtype=np.float64)
    d = np.linalg.norm(lidar_xyz - c, axis=1)
    inside = d < radius_m
    n = int(inside.sum())
    if n == 0:
        return {"matched": False, "pointsInRoi": 0, "verdict": "불일치", "deltaRangeM": None, "deltaBearingDeg": None}
    lid_roi = lidar_xyz[inside]
    lid_cent = lid_roi.mean(axis=0)
    lid_range = float(np.linalg.norm(lid_cent))
    lid_az = bearing_deg_xy(float(lid_cent[0]), float(lid_cent[1]))
    rr = float(radar_range_m) if radar_range_m is not None else float(np.linalg.norm(c))
    ra = float(radar_azimuth_deg) if radar_azimuth_deg is not None else bearing_deg_xy(float(c[0]), float(c[1]))
    delta_r = round(abs(rr - lid_range), 4)
    delta_bear = round(angle_diff_abs_deg(ra, lid_az), 4)
    matched = n >= 5
    verdict = "일치" if matched and delta_r < 15.0 and delta_bear < 5.0 else ("부분" if matched else "불일치")
    return {
        "matched": matched,
        "pointsInRoi": n,
        "deltaRangeM": delta_r,
        "deltaBearingDeg": delta_bear,
        "verdict": verdict,
    }


if lid_curr is not None and len(dets_curr_f) > 0:
    lid_xyz = lid_curr[:, :3]
    for rank, det in enumerate(dets_curr_f[:3]):
        lv = lidar_validate_cluster(
            lid_xyz,
            det["centroidM"],
            LIDAR_ROI_R,
            det["rangeM"],
            det["azimuthDeg"],
        )
        print(f"LiDAR 검증 rank{rank+1} {det['id']}:", lv)
else:
    print("LiDAR 없음 또는 후보 없음 — 검증 생략")

<a id="track"></a>
## 6. 연속 프레임 연관 · 속도 · heading · 8방위

- **연관:** 현재 후보(신뢰도 순)마다, 이전 프레임 centroid 집합에서 **거리 게이트 `G` 이내 최근접** 미사용 매칭 (그리디).  
- **속도:** $v = (c_t - c_{t-1}) / \Delta t$ (m/s).  
- **heading (수평):** $\mathrm{atan2}(v_y, v_x)$ in degrees.  
- **8방위:** 북=0°, 시계방향 또는 **ego 전방 기준** 중 하나를 택해 일관되게 정의. 아래는 **수학 각도(동쪽=0°, 반시계)** 가 아니라 **atan2(y,x) = 전방x·좌우y** 에 맞춘 **8분할 라벨**입니다.

In [ ]:
DT = float(os.environ.get("VOD_FRAME_DT_S", "0.1"))
TRACK_GATE_M = float(os.environ.get("VOD_TRACK_GATE_M", "12.0"))


def greedy_match_prev_curr(prev_centroids: np.ndarray, curr_dets: list[dict], gate_m: float) -> list[int | None]:
    if prev_centroids.size == 0:
        return [None] * len(curr_dets)
    used: set[int] = set()
    out: list[int | None] = []
    for cd in curr_dets:
        cc = np.array(cd["centroidM"], dtype=np.float64)
        best_j = None
        best_d = gate_m
        for j in range(prev_centroids.shape[0]):
            if j in used:
                continue
            dd = float(np.linalg.norm(prev_centroids[j] - cc))
            if dd < best_d:
                best_d = dd
                best_j = j
        if best_j is not None:
            used.add(best_j)
        out.append(best_j)
    return out


def heading_deg_from_vxy(vx: float, vy: float) -> float:
    return float(np.degrees(np.arctan2(vy, vx)))


def octant_label_ego(heading_deg: float) -> str:
    """ego: +x 전방=0°, +y 왼쪽=90° 근사 — 8방위 이름"""
    h = heading_deg % 360
    # 구간 중심: 전/좌전/좌/좌후/후/우후/우/우전
    edges = [(337.5, 22.5, "전방"), (22.5, 67.5, "좌전방"), (67.5, 112.5, "좌측"), (112.5, 157.5, "좌후방")]
    edges += [(157.5, 202.5, "후방"), (202.5, 247.5, "우후방"), (247.5, 292.5, "우측"), (292.5, 337.5, "우전방")]
    for a, b, name in edges:
        if a > b:
            if h >= a or h < b:
                return name
        else:
            if a <= h < b:
                return name
    return "?"


prev_centroids = np.array([d["centroidM"] for d in dets_prev_f], dtype=np.float64) if dets_prev_f else np.zeros((0, 3))
matches = greedy_match_prev_curr(prev_centroids, dets_curr_f, TRACK_GATE_M) if len(dets_curr_f) else []

tracks = []
for i, det in enumerate(dets_curr_f):
    pj = matches[i] if i < len(matches) else None
    if pj is None or prev_centroids.shape[0] == 0:
        tracks.append({**det, "motionMatched": False, "velocityMps": None, "speedMps": None, "headingDeg": None, "octant": None})
        continue
    c_prev = prev_centroids[pj]
    c_curr = np.array(det["centroidM"], dtype=np.float64)
    v = (c_curr - c_prev) / max(DT, 1e-6)
    spd = float(np.hypot(v[0], v[1]))
    hdg = heading_deg_from_vxy(float(v[0]), float(v[1]))
    tracks.append(
        {
            **det,
            "motionMatched": True,
            "velocityMps": [float(v[0]), float(v[1]), float(v[2])],
            "speedMps": round(spd, 4),
            "headingDeg": round(hdg, 2),
            "octant": octant_label_ego(hdg),
        }
    )

print("연관 성공 수:", sum(1 for t in tracks if t.get("motionMatched")))
for t in tracks[:5]:
    print(t["id"], "matched=", t.get("motionMatched"), "spd=", t.get("speedMps"), "oct=", t.get("octant"))

<a id="smooth"></a>
## 7. 추적 안정화 (게이트·스무딩)

- 이미 **거리 게이트**로 큰 점프 억제.  
- 추가로 **속도 폭**이 물리 한계를 넘으면 해당 스텝 속도를 이전 값과 혼합(간단 EMA).

In [ ]:
MAX_SPEED_MPS = float(os.environ.get("VOD_MAX_SPEED_MPS", "35.0"))
EMA_ALPHA = float(os.environ.get("VOD_VEL_EMA", "0.35"))


def stabilize_velocity(tracks: list[dict], prev_vel: dict[str, list[float]] | None) -> tuple[list[dict], dict[str, list[float]]]:
    prev_vel = prev_vel or {}
    new_prev: dict[str, list[float]] = {}
    out = []
    for t in tracks:
        tid = t["id"]
        if not t.get("motionMatched") or t.get("velocityMps") is None:
            out.append(t)
            continue
        v = np.array(t["velocityMps"], dtype=np.float64)
        spd = float(np.hypot(v[0], v[1]))
        if spd > MAX_SPEED_MPS:
            v = v * (MAX_SPEED_MPS / max(spd, 1e-6))
        if tid in prev_vel:
            pv = np.array(prev_vel[tid], dtype=np.float64)
            v = EMA_ALPHA * v + (1 - EMA_ALPHA) * pv
        new_prev[tid] = [float(v[0]), float(v[1]), float(v[2])]
        spd2 = float(np.hypot(v[0], v[1]))
        hdg2 = heading_deg_from_vxy(float(v[0]), float(v[1]))
        out.append({**t, "velocityMps": new_prev[tid], "speedMps": round(spd2, 4), "headingDeg": round(hdg2, 2), "octant": octant_label_ego(hdg2)})
    return out, new_prev


tracks_s, _ = stabilize_velocity(tracks, None)
print("스무딩 예시(1차):", tracks_s[0] if tracks_s else "—")

<a id="risk"></a>
## 8. 규칙 기반 위험도 · 미래 샘플 (ego 기준)

백엔드 `_rule_based_risk`와 동일 가중치:
- 근접(거리 정규화), 도플러 접근 항, 수평 속도 항, **기지(원점) 방향으로의 진행 성분**.

In [ ]:
R_REF = float(os.environ.get("VOD_RISK_RANGE_REF_M", "120.0"))


def rule_based_risk(range_m: float, doppler_mps: float, centroid: np.ndarray, velocity_mps: np.ndarray | None) -> dict:
    d_norm = max(0.0, min(1.0, 1.0 - float(range_m) / max(R_REF, 1.0)))
    approach = 1.0 if doppler_mps <= 0 else max(0.0, 1.0 - min(float(doppler_mps), 15.0) / 15.0 * 0.85)
    toward_ego = 0.0
    if velocity_mps is not None and velocity_mps.shape[0] >= 2:
        spd_xy = float(np.hypot(float(velocity_mps[0]), float(velocity_mps[1])))
        spd_score = min(1.0, spd_xy / 20.0)
        cxy = centroid[:2].astype(np.float64)
        rad = float(np.linalg.norm(cxy))
        if rad > 0.5 and spd_xy > 0.08:
            to_origin = -cxy / rad
            vxy = velocity_mps[:2].astype(np.float64)
            vxy = vxy / (np.linalg.norm(vxy) + 1e-9)
            toward_ego = max(0.0, float(np.dot(vxy, to_origin)))
    else:
        spd_score = min(1.0, abs(float(doppler_mps)) / 12.0) * 0.6
    score = 0.32 * d_norm + 0.28 * approach + 0.22 * spd_score + 0.18 * toward_ego
    score = max(0.0, min(1.0, float(score)))
    level = "낮음" if score < 0.35 else ("중간" if score < 0.65 else "높음")
    return {"score": round(score, 4), "level": level, "factors": {"d_norm": round(d_norm, 4), "approach": round(approach, 4)}}


def future_samples(centroid: list[float], v: np.ndarray | None, horizon_s: float, n: int) -> np.ndarray:
    n = max(2, int(n))
    c = np.array(centroid, dtype=np.float64)
    if v is None:
        return np.stack([c] * n)
    ts = np.linspace(0, horizon_s, n)
    return np.stack([c + v * t for t in ts])


primary = tracks_s[0] if tracks_s else None
if primary:
    cen = np.array(primary["centroidM"], dtype=np.float64)
    vm = np.array(primary["velocityMps"], dtype=np.float64) if primary.get("velocityMps") else None
    risk = rule_based_risk(primary["rangeM"], primary["dopplerMps"], cen, vm)
    fut = future_samples(primary["centroidM"], vm, horizon_s=3.0, n=10)
    print("1위 후보 위험도:", risk)
    print("미래 샘플 첫/끝:", fut[0], fut[-1])
else:
    print("후보 없음")

<a id="viz"></a>
## 9. 시각화 (BEV) 및 JSON 요약

- **BEV:** $x$–$y$ 평면에 원시 점(회색) + 클러스터 centroid(색) + (있으면) 미래 궤적

In [ ]:
primary_viz = tracks_s[0] if tracks_s else None
fig, ax = plt.subplots(figsize=(8, 8))
if pts_curr_f.shape[0] > 0:
    ax.scatter(pts_curr_f[:, 0], pts_curr_f[:, 1], s=1, c="#94a3b8", alpha=0.35, label="radar pts")
for t in tracks_s:
    c = t["centroidM"]
    ax.scatter(c[0], c[1], s=40 + t["clusterSize"], alpha=0.85, label=None)
    ax.annotate(t["id"], (c[0], c[1]), fontsize=8)
if primary_viz and primary_viz.get("velocityMps"):
    vm = np.array(primary_viz["velocityMps"], dtype=np.float64)
    fut = future_samples(primary_viz["centroidM"], vm, 3.0, 16)
    ax.plot(fut[:, 0], fut[:, 1], "c-", linewidth=2, label="외삽 궤적")
ax.set_aspect("equal")
ax.set_xlabel("x (전방, m)")
ax.set_ylabel("y (m)")
ax.set_title("FMCW VoD — BEV (현재 프레임)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
plt.show()

In [ ]:
summary = {
    "stems": {"prev": stem_prev, "curr": stem_curr},
    "preprocess": {"max_range_m": MAX_RANGE_M, "min_x": MIN_X_M, "z_abs_max": Z_ABS_MAX},
    "dbscan": {"eps": DBSCAN_EPS, "min_samples": DBSCAN_MIN_SAMPLES},
    "track": {"dt_s": DT, "gate_m": TRACK_GATE_M},
    "n_detections": len(dets_curr_f),
    "tracks": [{k: v for k, v in t.items() if k != "velocityMps" or v is not None} for t in tracks_s[:8]],
}
if primary and primary.get("velocityMps"):
    cen = np.array(primary["centroidM"], dtype=np.float64)
    vm = np.array(primary["velocityMps"], dtype=np.float64)
    summary["primary_risk"] = rule_based_risk(primary["rangeM"], primary["dopplerMps"], cen, vm)

print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

if pd is not None and tracks_s:
    rows = []
    for t in tracks_s:
        rows.append(
            {
                "id": t["id"],
                "rangeM": t["rangeM"],
                "conf": t["confidence"],
                "n": t["clusterSize"],
                "matched": t.get("motionMatched"),
                "speed": t.get("speedMps"),
                "heading": t.get("headingDeg"),
                "octant": t.get("octant"),
            }
        )
    display(pd.DataFrame(rows))
else:
    print("(pandas 없음 — 표 생략)")

---
### 부록: 프로젝트 문서와의 매핑

| 문서 항목 | 노트북 섹션 |
|-----------|-------------|
| VoD 3+1D 포인트, x,y,z,RCS,v_r,v_r_comp,time | §1 스키마 |
| 이상치·ROI·속도 보정 활용 | §2 `preprocess_radar` |
| DBSCAN 후보·신뢰도 | §3–4 |
| LiDAR·동기 검증 | §5 |
| 연속 프레임·최근접 연관·파생변수 | §6 |
| 게이트·스무딩 | §7 |
| 규칙 위험도·진행구역 외삽 | §8 |
| 지도 연동용 좌표는 웹에서 polar 변환 | §8–9 (ego 상대 → 백엔드 `polarToLatLng`와 동일 개념) |

**카메라/YOLO** 검증은 `ultralytics` 의존이 필요하므로 본 노트북에서는 생략했습니다. API 통합은 `ai-inference/main.py`의 `infer_vod_radar_fusion`을 참고하세요.